# 3 Sep

# Parsing and checking out the coconut_with_cids.csv so I can ensure completeness for the DB and remove things I dont need like the descriptors and the organisms

## Getting an idea of whats here what is missing and what is distinct

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

columns = coco.columns
columns.to_list()

entries = []
for col in columns:
    num_entries = len(coco[col])
    missing = coco[col].isnull().sum()
    distinct = coco[col].nunique()
    entries.append([col,num_entries,missing,distinct])

df_out = pd.DataFrame(entries,columns=['column_name','number_of_entries','missing_entries','distinct_entries'])
print(df_out)

                         column_name  number_of_entries  missing_entries  \
0                         identifier             738827                0   
1                   canonical_smiles             738827                0   
2                     standard_inchi             738827                0   
3                 standard_inchi_key             738827                0   
4                               name             738827           375404   
5                         iupac_name             738827            74593   
6                   annotation_level             738827                0   
7                   total_atom_count             738827                0   
8                   heavy_atom_count             738827                0   
9                   molecular_weight             738827                0   
10            exact_molecular_weight             738827                0   
11                 molecular_formula             738827                0   
12          

## Checking the coconut paper (old paper to be fair) claim that all compounds have a name or iupac

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name) and pd.isna(row.synonyms):
        count +=   1

print(f'There are {count} entries with no obvious name')

There are 0 entries with no obvious name


## Giving each compound a name and dropping some of the columns I don't need

In [3]:
# im lazy just printing column names so I can copy paste
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', low_memory=False)

cols = coco.columns
cols = cols.to_list()
print(cols)

['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name', 'annotation_level', 'total_atom_count', 'heavy_atom_count', 'molecular_weight', 'exact_molecular_weight', 'molecular_formula', 'alogp', 'topological_polar_surface_area', 'rotatable_bond_count', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski', 'lipinski_rule_of_five_violations', 'aromatic_rings_count', 'qed_drug_likeliness', 'formal_charge', 'fractioncsp3', 'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar', 'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework', 'np_likeness', 'chemical_class', 'chemical_sub_class', 'chemical_super_class', 'direct_parent_classification', 'np_classifier_pathway', 'np_classifier_superclass', 'np_classifier_class', 'np_classifier_is_glycoside', 'organisms', 'collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid']


In [4]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'canonical_smiles', 'standard_inchi', 'standard_inchi_key', 'name', 'iupac_name',
                                                                                               'np_likeness','collections', 'dois', 'synonyms', 'cas', 'coconut_id', 'cid'], low_memory=False)
count = 0
for idx, row in coco.iterrows():
    if pd.isna(row.name) and pd.isna(row.iupac_name):
        count += 1
print(count)


0


## for the above I was going to sort out naming but apparently no entry is missing its iupac and name but it looks like the iupacs missinga are not necessarily all unresolveable so that is the job here but I am writing this so it can be done on wonko

- about 26% of the compounds that do nothave an IUPAC do have a CID so I will just send the ones with cids to wonko to try
- first code block generates the file for wonko the second will be the actual script 

In [1]:
import pandas as pd

coco = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_with_cids.csv', usecols=['identifier', 'iupac_name', 'cid'], low_memory=False)

df = coco[coco["iupac_name"].isna() | coco["iupac_name"].astype(str).str.strip().eq("")].copy()

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_compounds_need_iupac_for_wonko.csv',index=False)

In [ ]:
import pandas as pd
import time
import pubchempy as pcp

def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'

def progress_bar(count, total):
    bar_length = 40
    filled_length = int(bar_length * count // total)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    out = f'\rProgress: |{bar}| {count}/{total} ({(count/total)*100:.2f}%)'
    return out

in_path = '/nlustre/users/nathanc/coconut_compounds_need_iupac_for_wonko.csv'
results_path  = '/nlustre/users/nathanc/coconut_now_with_iupac_from_pubchem.csv'
error_path = '/nlustre/users/nathanc/iupac_getter_errors.csv'

df = pd.read_csv(in_path)
err = []
count = 0
length = len(df)

for idx, row in df.iterrows():
        if pd.notna(row.cid):
            time.sleep(0.2)
            cid = int(row.cid) 
            
            iupac, errorz = get_iupac(cid)

            if iupac:
                df.at[idx,'iupac_name'] = iupac
            else:
                err.append([cid,errorz])
        count += 1
        
        if count % 1000 == 0:
                df.to_csv(results_path,index=False)
                errors = pd.DataFrame(err, columns = ['cid','err'])
                errors.to_csv(error_path, index=False)
        print(progress_bar(count,length))

df.to_csv(results_path, index=False)
errors = pd.DataFrame(err, columns = ['cid','err'])
errors.to_csv(error_path, index=False)


# 4 Sep

## Wonko made a nice file and error file for the iupac getter stuff. (coconut_full/iupac_getter_errors.csv,coconut_full/coconut_now_with_iupac_from_pubchem.csv)
- just checking the error file to see whats up

In [5]:
import pandas as pd

errors = pd.read_csv('/home/school/masters/Scripts/coconut_full/iupac_getter_errors.csv')

error_types = errors['err'].drop_duplicates()
error_nums = errors['err'].value_counts()

df_out = pd.DataFrame({'error_type': error_types, 'count': error_nums.values})
print(errors.shape[0])
print(df_out)



2537
                                            error_type  count
0                                             no iupac   2519
169                                 retry uncsucessful     15
215  <urlopen error [Errno -2] Name or service not ...      3


## retrying the 18 that didn't come back as no iupac

In [ ]:


def get_iupac(cid):
    for k in range(5):
        try:
            compound = pcp.get_compounds(int(cid),'cid')
            if compound:
                iupac = compound[0].iupac_name
                if iupac:
                    return iupac, None
                else:
                    return None, 'no iupac'
            else: 
                return None, 'didnt work not on pubchem'
        except Exception as e:
            error_msg = str(e)
            if '502' in error_msg or '503' in error_msg or '504' in error_msg or 'bad gateway' in error_msg.lower() or 'timeout' in error_msg.lower():
                time.sleep(5)
            else:
                return None, str(e)
    return None, 'retry uncsucessful'